In [ ]:
import polars as pl
from fof8_core.loader import FOF8Loader
from fof8_core.targets import calculate_season_av, get_career_value_metrics
from fof8_core.targets_replacements import strategy_hybrid_baseline, strategy_25th_percentile

# Initialize loader
loader = FOF8Loader("../fof8-gen/data/raw", "DRAFT003")

# 1. Get raw season production (AV)
lf_av = calculate_season_av(loader.scan_file("player_record.csv"))
# 2. Calculate VORP using the hybrid strategy
# The function now returns the full dataset with "Replacement_AV" and "Season_VORP"
df_vorp = strategy_hybrid_baseline(lf_av).collect()
print(f"Season-by-season VORP table created with {len(df_vorp)} rows.")
# Inspect results - Replacement_AV and Season_VORP are now included by default
print(df_vorp.select(["Year", "Position_Group", "Season_AV", "Replacement_AV", "Season_VORP"]).head())

In [ ]:
# 2. Calculate Season VORP for different strategies
# Strategies now return the full player dataframe with Season_VORP and Replacement_AV
df_hybrid = strategy_hybrid_baseline(lf_av).collect()
df_legacy = strategy_25th_percentile(lf_av).collect()

# Compare the baseline for a specific position
year_filter = 2021
pos = "QB"

# For QBs (Ironmen), the replacement level is still relatively static
hybrid_val = df_hybrid.filter((pl.col('Year')==year_filter) & (pl.col('Position_Group')==pos))['Replacement_AV'].mean()
legacy_val = df_legacy.filter((pl.col('Year')==year_filter) & (pl.col('Position_Group')==pos))['Replacement_AV'].mean()

print(f"--- {year_filter} {pos} Mean Replacement Level ---")
print(f"Hybrid (Rate-based): {hybrid_val:.2f}")
print(f"Legacy (25th Percentile): {legacy_val:.2f}")


In [ ]:
# 3. Get final career metrics (uses Hybrid strategy by default)
df_career = get_career_value_metrics(loader)

# Show the top career value earners
print("Top 10 Career VORP Leaders (Hybrid Baseline):")
print(df_career.sort("Career_VORP", descending=True).head(10))


In [ ]:
# Compare top performers using a different strategy (e.g. strategy_n_plus_one)
from fof8_core.targets_replacements import strategy_n_plus_one

df_career_n1 = get_career_value_metrics(loader, strategy=strategy_n_plus_one)

print("Top Career VORP Leaders (N+1 Baseline):")
print(df_career_n1.sort("Career_VORP", descending=True).head(5))


In [ ]:
# 1. Get raw season production (AV)
lf_av = calculate_season_av(loader.scan_file("player_record.csv"))

# 2. Calculate VORP using the hybrid strategy
# The function now handles the entire calculation and returns the full dataset
df_vorp = strategy_hybrid_baseline(lf_av).collect()

print(f"Season-by-season VORP table created with {len(df_vorp)} rows.")
# Inspect results
print(df_vorp.select(["Year", "Position_Group", "Season_AV", "Replacement_AV", "Season_VORP"]).head())


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Filter for active seasons (VORP > -5 to remove massive outliers)
plt.figure(figsize=(12, 6))
sns.boxplot(data=df_vorp.to_pandas(), x="Position_Group", y="Season_VORP")
plt.title("Season VORP Distribution by Position (Hybrid Baseline)")
plt.xticks(rotation=45)
plt.show()


In [ ]:
# Trend of Replacement AV over time for core positions
trend_data = df_replacement.filter(
    pl.col("Position_Group").is_in(["QB", "WR", "T", "DE", "CB"])
).to_pandas()

plt.figure(figsize=(12, 6))
sns.lineplot(data=trend_data, x="Year", y="Replacement_AV", hue="Position_Group")
plt.title("Replacement Level Stability Over Time")
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
# Join the two results
comparison = (
    get_career_value_metrics(loader, strategy=strategy_hybrid_baseline).select(["Player_ID", "Career_VORP"]).rename({"Career_VORP": "Hybrid"})
    .join(
        get_career_value_metrics(loader, strategy=strategy_25th_percentile).select(["Player_ID", "Career_VORP"]).rename({"Career_VORP": "Legacy"}),
        on="Player_ID"
    )
).to_pandas()

plt.figure(figsize=(8, 8))
plt.scatter(comparison["Legacy"], comparison["Hybrid"], alpha=0.5, s=10)
plt.plot([0, 500], [0, 500], color='red', linestyle='--') # 45-degree line
plt.xlabel("Legacy Career VORP")
plt.ylabel("Hybrid Career VORP")
plt.title("How the Hybrid Strategy Re-evaluates Careers")
plt.show()


In [ ]:
# Look at a single team's QB room in 2021
team_id = 0 # Adjust as needed
qb_room = df_vorp.filter((pl.col("Year")==2021) & (pl.col("Team")==team_id) & (pl.col("Position_Group")=="QB"))
print(qb_room.select(["Player_ID", "Experience", "Salary_Year_1", "Season_AV", "Replacement_AV", "Season_VORP"]))


In [ ]:
import polars as pl
import hvplot.pandas
import holoviews as hv

# 1. Load player records across all years
# This scans the player_record.csv files for every year in the simulation
lf_records = loader.scan_file("player_record.csv")

# 2. Aggregate to find the career-high HOF points for every player
# We also keep the Position_Group (taking the last one they were assigned)
df_hof = (
    lf_records
    .group_by("Player_ID")
    .agg([
        pl.col("Hall_Of_Fame_Points").max().alias("Career_HOF_Points"),
        pl.col("Position_Group").last().alias("Position_Group")
    ])
    .filter(pl.col("Career_HOF_Points") > 0) # Focus on players with actual recognition
    .collect()
)

# 3. Create a violin plot to see the distribution by Position Group
# Violin plots are great for showing density and outliers across categories
plot = df_hof.to_pandas().hvplot.violin(
    y='Career_HOF_Points', 
    by='Position_Group', 
    height=500, 
    width=900,
    title="Hall of Fame Points Distribution by Position Group",
    ylabel="HOF Points",
    xlabel="Position",
    rot=45,
    padding=0.1,
    cmap='Category20'
)

# 4. Display a summary table of the elite outliers
print("--- Elite HOF Point Leaders (Top 10) ---")
print(df_hof.sort("Career_HOF_Points", descending=True).head(10))

plot


In [ ]:
df_hof.sort("Career_HOF_Points", descending=False).head(10)

In [ ]:
# 1. Calculate Positional Z-Scores
# We calculate mean and std within each Position_Group to see who deviates most from the norm
df_hof_z = (
    df_hof # Using the df_hof from the previous cell (players with points > 0)
    .with_columns([
        (
            (pl.col("Career_HOF_Points") - pl.col("Career_HOF_Points").mean().over("Position_Group")) / 
            pl.col("Career_HOF_Points").std().over("Position_Group").fill_null(1.0) # Avoid div by zero
        ).alias("HOF_Z_Score")
    ])
)

# 2. Plot the Z-Score Distribution
# Since Z-scores are normalized, all positions will now be centered around 0
z_plot = df_hof_z.to_pandas().hvplot.violin(
    y='HOF_Z_Score', 
    by='Position_Group', 
    height=500, 
    width=900,
    title="Relative Greatness: HOF Point Z-Scores by Position",
    ylabel="Z-Score (Standard Deviations from Positional Mean)",
    xlabel="Position",
    rot=45,
    padding=0.1,
    cmap='viridis'
)

# 3. Identify the "True Outliers" (Players more than 3 standard deviations above their peers)
print("--- Positional 'God-Tier' Outliers (Z > 3.0) ---")
print(
    df_hof_z
    .filter(pl.col("HOF_Z_Score") > 3.0)
    .sort("HOF_Z_Score", descending=True)
    .select(["Position_Group", "Career_HOF_Points", "HOF_Z_Score"])
    .head(15)
)

z_plot
